# Дослідження популярності мов програмування (2020–2026 рр.)
**Курсова робота з дисципліни «Мови програмування для аналізу даних»**  
**Тема № 12.** Дослідження популярності мов програмування  
**Студент:** Литвин Назар Андрійович, група 401-ТН, спеціальність 122 «Комп’ютерні науки»  
**Керівник:** старший викладач Гайтан Олена Миколаївна

---

## 1. Аналіз предметної області та постановка завдання
Мета дослідження — комплексно оцінити глобальну й національну (зокрема українську) популярність мов програмування за 2020–2026 рр. на основі фактичної активності розробників у репозиторіях GitHub та довідково-екосистемних метрик PLDB.

### Дослідницькі питання (RQ)
- **RQ1:** Які мови програмування лідирують у світі станом на 2026 Q2 за кількістю активних розробників і часткою в категорії `programming`, та яку частку становлять утиліти автоматизації (`Shell`, `Dockerfile`, `Makefile`)?
- **RQ2:** Як змінилася популярність провідних мов програмування у динаміці з 2020 Q1 до 2026 Q2 та які мови продемонстрували найвищу кратність зростання?
- **RQ3:** Чим відрізняється структура попиту на мови програмування в Україні (`UA`) від загальносвітового рейтингу?
- **RQ4:** Як співвідноситься інтенсивність використання мов різних поколінь (`language_era`) між економіками різного масштабу (`dev_market_tier`)?
- **RQ5:** Якою є форма розподілу активності й частка статистичних викидів за правилом IQR?
- **RQ6:** Наскільки тісно реальна активність розробників на GitHub пов’язана з оцінками кількості користувачів, вакансій, книг, відвідуваності Wikipedia та віком мови у базі PLDB?

In [1]:
import json
import pathlib
import sys

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

# Налаштування шляхів і підключення допоміжних констант із src/
PROJECT_ROOT = pathlib.Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis_pipeline import (
    COUNTRY_META,
    LANGUAGE_ALIASES,
    PLDB_COLS,
    TOOLING_LANGUAGES,
    setup_plot_style,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

setup_plot_style()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print(f"Робоча директорія проєкту: {PROJECT_ROOT}")

Робоча директорія проєкту: D:\мпад\курсова робота


## 2. Завантаження та первинне дослідження структури даних
Використовуємо 4 пов’язані таблиці:
1. `languages.csv` — поквартальні дані GitHub Innovation Graph про кількість унікальних розробників (`num_pushers`) за мовами та країнами;
2. `developers.csv` — загальна кількість розробників у країнах за кварталами;
3. `repositories.csv` — загальна кількість репозиторіїв у країнах за кварталами;
4. `pldb.csv` — база знань Programming Language DataBase (рік появи, оцінка користувачів, вакансій, книг, Wikipedia, рейтинг PLDB).

> **Важливий методичний нюанс:** код країни Намібія за стандартом ISO 3166-1 alpha-2 має вигляд `"NA"`. За стандартного виклику `pd.read_csv()` бібліотека Pandas інтерпретує рядок `"NA"` як відсутнє значення `NaN`. Перевіримо це явище кількісно та перезавантажимо таблиці з параметрами `keep_default_na=False, na_values=[""]`.

In [2]:
# Демонстрація проблеми автоматичного розпізнавання коду Намібії 'NA' як NaN
for fname in ("languages.csv", "developers.csv", "repositories.csv"):
    tmp = pd.read_csv(RAW_DIR / fname)
    print(f"Стандартне читання {fname:16s} -> хибних NaN у iso2_code: {tmp['iso2_code'].isna().sum()}")

# Коректне завантаження без спотворення коду 'NA'
languages_raw = pd.read_csv(RAW_DIR / "languages.csv", keep_default_na=False, na_values=[""])
developers_raw = pd.read_csv(RAW_DIR / "developers.csv", keep_default_na=False, na_values=[""])
repositories_raw = pd.read_csv(RAW_DIR / "repositories.csv", keep_default_na=False, na_values=[""])
pldb_raw = pd.read_csv(RAW_DIR / "pldb.csv", low_memory=False)

for name, tbl in [
    ("languages_raw", languages_raw),
    ("developers_raw", developers_raw),
    ("repositories_raw", repositories_raw),
    ("pldb_raw", pldb_raw),
]:
    print(f"{name:18s}: shape={tbl.shape}, exact_duplicates={tbl.duplicated().sum()}")

print("\nПерші 5 рядків languages_raw:")
display(languages_raw.head())
print("\nІнформація про типи та структуру languages_raw:")
languages_raw.info()
print("\nУнікальні значення language_type:", languages_raw["language_type"].value_counts().to_dict())
print("Кількість унікальних мов:", languages_raw["language"].nunique(), "| кодів iso2:", languages_raw["iso2_code"].nunique())

Стандартне читання languages.csv    -> хибних NaN у iso2_code: 93
Стандартне читання developers.csv   -> хибних NaN у iso2_code: 26
Стандартне читання repositories.csv -> хибних NaN у iso2_code: 26


languages_raw     : shape=(190246, 6), exact_duplicates=0


developers_raw    : shape=(5888, 4), exact_duplicates=0
repositories_raw  : shape=(5866, 4), exact_duplicates=0
pldb_raw          : shape=(5159, 403), exact_duplicates=0

Перші 5 рядків languages_raw:


,num_pushers,language,language_type,iso2_code,year,quarter
0,2066,HTML,markup,AE,2020,1
1,1627,CSS,markup,AE,2020,1
2,288,Jupyter Notebook,markup,AE,2020,1
3,108,Vue,markup,AE,2020,1
4,1734,JavaScript,programming,AE,2020,1



Інформація про типи та структуру languages_raw:
<class 'pandas.DataFrame'>
RangeIndex: 190246 entries, 0 to 190245
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   num_pushers    190246 non-null  int64
 1   language       190246 non-null  str  
 2   language_type  190246 non-null  str  
 3   iso2_code      190246 non-null  str  
 4   year           190246 non-null  int64
 5   quarter        190246 non-null  int64
dtypes: int64(3), str(3)
memory usage: 8.7 MB

Унікальні значення language_type: {'programming': 152073, 'markup': 37187, 'data': 673, 'prose': 313}
Кількість унікальних мов: 405 | кодів iso2: 188


## 3. Попереднє опрацювання даних та обґрунтування очищення
На етапі перевірки якості виконуються такі кроки:
1. **Відокремлення агрегату `'EU'`:** у таблицях GitHub присутній наднаціональний код `'EU'` (Євросоюз — 5 510 рядків у `languages.csv`, по 26 рядків у `developers.csv` і `repositories.csv`). Оскільки окремі країни ЄС (`DE`, `FR`, `PL` тощо) вже включені до таблиць, збереження `'EU'` у спільній вибірці призвело б до подвійного підрахунку. Відокремлюємо `'EU'` до окремої таблиці `languages_eu`.
2. **Очищення `pldb.csv`:** відбираємо 15 релевантних ознак, приводимо числові поля (`pd.to_numeric`), обнулюємо історично-математичні записи з `appeared < 1940` (43 записи, не належать до обчислювальних мов), а також усуваємо 85 дублікатів за нормалізованою назвою `language_key` зі збереженням запису з найкращим рейтингом `rank`.
3. **Аудит доменних обмежень:** мінімальне значення `num_pushers` становить `101` (поріг конфіденційності GitHub Innovation Graph `> 100` розробників на країну-квартал).

In [3]:
eu_lang_cnt = (languages_raw["iso2_code"] == "EU").sum()
languages_eu = languages_raw[languages_raw["iso2_code"] == "EU"].copy()
languages_clean = languages_raw[languages_raw["iso2_code"] != "EU"].drop_duplicates().copy()
developers_clean = developers_raw[developers_raw["iso2_code"] != "EU"].drop_duplicates().copy()
repositories_clean = repositories_raw[repositories_raw["iso2_code"] != "EU"].drop_duplicates().copy()

for col in ("num_pushers", "year", "quarter"):
    languages_clean[col] = pd.to_numeric(languages_clean[col], errors="coerce").astype("int64")

pldb_sub = pldb_raw[PLDB_COLS].copy()
print("Пропуски у вибраних 15 полях PLDB до очищення:")
display(pldb_sub.isna().sum().to_frame("Пропусків").T)

for col in [
    "appeared", "rank", "pldbScore", "foundationScore",
    "numberOfUsersEstimate", "numberOfJobsEstimate",
    "githubLanguage_repos", "wikipedia_dailyPageViews",
    "bookCount", "paperCount",
]:
    pldb_sub[col] = pd.to_numeric(pldb_sub[col], errors="coerce")

invalid_years = ((pldb_sub["appeared"] < 1940) | (pldb_sub["appeared"] > 2026)).sum()
pldb_sub.loc[(pldb_sub["appeared"] < 1940) | (pldb_sub["appeared"] > 2026), "appeared"] = np.nan

pldb_sub["language_key"] = pldb_sub["name"].astype(str).str.strip().str.lower()
pldb_sub = pldb_sub.sort_values("rank", ascending=True, na_position="last")
dup_keys = pldb_sub["language_key"].duplicated().sum()
pldb_clean = pldb_sub.drop_duplicates(subset=["language_key"], keep="first").copy()

print(f"Відокремлено записів 'EU' у languages: {eu_lang_cnt}; залишилось по суверенних країнах: {len(languages_clean):,}")
print(f"PLDB: обнулено {invalid_years} записів із appeared < 1940; видалено {dup_keys} дублікатів ключів; фінал: {len(pldb_clean):,}")

Пропуски у вибраних 15 полях PLDB до очищення:

,id,name,appeared,primaryTag,rank,pldbScore,foundationScore,numberOfUsersEstimate,numberOfJobsEstimate,githubLanguage_repos,wikipedia_dailyPageViews,bookCount,paperCount,isOpenSource,creators
Пропусків,0,0,0,0,0,0,0,0,0,4669,3494,0,0,3528,3515


Відокремлено записів 'EU' у languages: 5510; залишилось по суверенних країнах: 184,736
PLDB: обнулено 43 записів із appeared < 1940; видалено 85 дублікатів ключів; фінал: 5,074


## 4. Об’єднання таблиць (`merge`) та створення нових аналітичних ознак
Виконуємо три обґрунтовані `merge()`:
1. `languages_clean` + `developers_clean` за ключем `['iso2_code', 'year', 'quarter']` (`100.0%` збіг);
2. Результат + `repositories_clean` за ключем `['iso2_code', 'year', 'quarter']` (`100.0%` збіг);
3. Результат + `pldb_clean` за нормалізованим ключем `language_key` з урахуванням словника аліасів (`shell -> bash`, `makefile -> make`, `assembly -> assembly language`, `plpgsql -> pl/pgsql`).

Після об’єднання формуємо **7 нових аналітичних ознак**: `period_date`, `pushers_per_1000_devs`, `repos_per_dev`, `language_share_pct`, `language_age`, `language_era` (`pd.cut`), `dev_market_tier` (`pd.cut`), а також ознаку `is_general_purpose`.

In [4]:
df_m1 = languages_clean.merge(developers_clean, on=["iso2_code", "year", "quarter"], how="left", validate="many_to_one")
df_m2 = df_m1.merge(repositories_clean, on=["iso2_code", "year", "quarter"], how="left", validate="many_to_one")

df_m2["language_key"] = df_m2["language"].astype(str).str.strip().str.lower().replace(LANGUAGE_ALIASES)
pldb_merge_cols = [
    "language_key", "appeared", "primaryTag", "rank", "pldbScore",
    "foundationScore", "numberOfUsersEstimate", "numberOfJobsEstimate",
    "githubLanguage_repos", "wikipedia_dailyPageViews", "bookCount", "paperCount",
]
df = df_m2.merge(pldb_clean[pldb_merge_cols], on="language_key", how="left", validate="many_to_one")

print(f"Покриття merge 1 (developers):   {df['developers'].notna().mean()*100:.2f}%")
print(f"Покриття merge 2 (repositories): {df['repositories'].notna().mean()*100:.2f}%")
print(f"Покриття merge 3 (PLDB за рядками):  {df['appeared'].notna().mean()*100:.2f}%")
print(f"Покриття merge 3 (PLDB за обсягом pushers): {df.loc[df['appeared'].notna(), 'num_pushers'].sum()/df['num_pushers'].sum()*100:.2f}%")

# Інженерія аналітичних ознак
quarter_start_month = (df["quarter"] - 1) * 3 + 1
df["period_date"] = pd.to_datetime(dict(year=df["year"], month=quarter_start_month, day=1))
df["period_label"] = df["year"].astype(str) + "-Q" + df["quarter"].astype(str)
df["pushers_per_1000_devs"] = (df["num_pushers"] / df["developers"]) * 1000.0
df["repos_per_dev"] = df["repositories"] / df["developers"]

grp_totals = df.groupby(["iso2_code", "year", "quarter", "language_type"])["num_pushers"].transform("sum")
df["language_share_pct"] = (df["num_pushers"] / grp_totals) * 100.0
df["language_age"] = df["year"] - df["appeared"]
df["language_era"] = pd.cut(
    df["appeared"],
    bins=[1940, 1989, 2004, 2014, 2026],
    labels=["До 1990 (класичні)", "1990–2004 (веб-епоха)", "2005–2014 (сучасні)", "2015+ (новітні)"],
    include_lowest=True,
)
df["dev_market_tier"] = pd.cut(
    df["developers"],
    bins=[0, 50_000, 500_000, 2_000_000, 1e9],
    labels=["Малі (<50 тис.)", "Середні (50–500 тис.)", "Великі (0.5–2 млн)", "Глобальні лідери (>2 млн)"],
)
df["is_general_purpose"] = (df["language_type"] == "programming") & (~df["language"].isin(TOOLING_LANGUAGES))
df["country_name_uk"] = df["iso2_code"].map(lambda c: COUNTRY_META[c][0] if c in COUNTRY_META else f"Інші ({c})")

print("Фінальний розмір збагаченої таблиці:", df.shape)
display(df[["period_label", "iso2_code", "language", "num_pushers", "pushers_per_1000_devs", "language_share_pct", "language_age", "language_era"]].head(6))

Покриття merge 1 (developers):   100.00%
Покриття merge 2 (repositories): 100.00%
Покриття merge 3 (PLDB за рядками):  94.41%
Покриття merge 3 (PLDB за обсягом pushers): 99.11%


Фінальний розмір збагаченої таблиці: (184736, 30)


,period_label,iso2_code,language,num_pushers,pushers_per_1000_devs,language_share_pct,language_age,language_era
0,2020-Q1,AE,HTML,2066,21.703961,50.525801,29.0,1990–2004 (веб-епоха)
1,2020-Q1,AE,CSS,1627,17.092132,39.789680,24.0,1990–2004 (веб-епоха)
2,2020-Q1,AE,Jupyter Notebook,288,3.025528,7.043287,6.0,2005–2014 (сучасні)
3,2020-Q1,AE,Vue,108,1.134573,2.641233,6.0,2005–2014 (сучасні)
4,2020-Q1,AE,JavaScript,1734,18.216199,23.546985,25.0,1990–2004 (веб-епоха)
5,2020-Q1,AE,Python,816,8.572329,11.080934,29.0,1990–2004 (веб-епоха)


## 5. Статистичний аналіз, групування, ранжування та зведені таблиці
### 5.1 Описова статистика та аудит викидів за правилом IQR

In [5]:
stat_cols = ["num_pushers", "pushers_per_1000_devs", "developers", "repositories", "language_share_pct", "language_age"]
stats_summary = df[stat_cols].describe().T
stats_summary["IQR"] = stats_summary["75%"] - stats_summary["25%"]
stats_summary["skew"] = df[stat_cols].skew()
stats_summary["outliers_IQR_cnt"] = [
    int((df[c] > (stats_summary.loc[c, "75%"] + 1.5 * stats_summary.loc[c, "IQR"])).sum())
    for c in stat_cols
]
display(stats_summary.round(2))

,count,mean,std,min,25%,50%,75%,max,IQR,skew,outliers_IQR_cnt
num_pushers,184736.0,3052.71,18641.93,101.0,183.00,394.00,1282.00,1424465.00,1099.00,28.26,26115
pushers_per_1000_devs,184736.0,3.91,8.06,0.0,0.19,0.73,3.38,169.98,3.18,3.68,23726
developers,184736.0,2518503.92,5039080.18,1966.0,246969.00,784623.00,2180029.00,35064239.00,1933060.00,3.67,19957
repositories,184736.0,6389241.12,13155669.42,2320.0,569836.00,1828669.00,5526230.00,81273455.00,4956394.00,3.57,18266
language_share_pct,184736.0,4.78,12.12,0.0,0.15,0.61,3.15,100.00,3.00,4.44,24498
language_age,174405.0,25.08,13.57,0.0,14.00,24.00,34.00,76.00,20.00,0.70,1467


### 5.2 Рейтинг Топ-15 мов у світі (2026 Q2) та порівняння з Україною (`UA`)

In [6]:
q_latest = df[(df["year"] == 2026) & (df["quarter"] == 2)].copy()
prog_latest = q_latest[q_latest["language_type"] == "programming"].copy()

global_prog_2026 = (
    prog_latest.groupby(["language", "is_general_purpose"], as_index=False)
    .agg(global_pushers=("num_pushers", "sum"), countries=("iso2_code", "nunique"), mean_intensity=("pushers_per_1000_devs", "mean"))
    .sort_values("global_pushers", ascending=False)
)
global_prog_2026["global_rank"] = global_prog_2026["global_pushers"].rank(ascending=False, method="min").astype(int)
global_prog_2026["global_share_pct"] = global_prog_2026["global_pushers"] / global_prog_2026["global_pushers"].sum() * 100.0

print("Топ-15 мов програмування у світі (2026 Q2):")
display(global_prog_2026.nlargest(15, "global_pushers").round(2))

ua_latest = prog_latest[prog_latest["iso2_code"] == "UA"].copy()
ua_latest["ua_rank"] = ua_latest["num_pushers"].rank(ascending=False, method="min").astype(int)
ua_latest["ua_share_pct"] = ua_latest["num_pushers"] / ua_latest["num_pushers"].sum() * 100.0
ua_comp = ua_latest.merge(global_prog_2026[["language", "global_rank", "global_share_pct"]], on="language", how="left")
ua_comp["rank_diff"] = ua_comp["global_rank"] - ua_comp["ua_rank"]

print("\nТоп-12 мов програмування в Україні (UA, 2026 Q2):")
display(ua_comp.nlargest(12, "num_pushers")[["ua_rank", "language", "num_pushers", "ua_share_pct", "pushers_per_1000_devs", "global_rank", "rank_diff"]].round(2))

Топ-15 мов програмування у світі (2026 Q2):


,language,is_general_purpose,global_pushers,countries,mean_intensity,global_rank,global_share_pct
137,JavaScript,True,6162445,182,30.77,1,19.81
219,Python,True,4509915,166,18.64,2,14.50
278,TypeScript,True,3025534,165,15.34,3,9.73
252,Shell,False,2353843,153,10.08,4,7.57
79,Dockerfile,False,1819686,151,8.65,5,5.85
136,Java,True,1062585,142,5.28,6,3.42
39,C++,True,930908,140,4.36,7,2.99
37,C,True,874642,140,4.15,8,2.81
164,Makefile,False,791943,126,3.45,9,2.55
27,Batchfile,False,706986,135,3.16,10,2.27



Топ-12 мов програмування в Україні (UA, 2026 Q2):


,ua_rank,language,num_pushers,ua_share_pct,pushers_per_1000_devs,global_rank,rank_diff
0,1,JavaScript,41928,20.34,22.97,1,0
1,2,Python,29125,14.13,15.96,2,0
2,3,TypeScript,19942,9.67,10.93,3,0
3,4,Dockerfile,14453,7.01,7.92,5,1
4,5,Shell,14446,7.01,7.92,4,-1
5,6,C#,8040,3.90,4.41,13,7
6,7,C++,7154,3.47,3.92,7,0
7,8,Java,6313,3.06,3.46,6,-2
8,9,C,5674,2.75,3.11,8,-1
9,10,Makefile,5096,2.47,2.79,9,-1


### 5.3 Групування, зведені таблиці (`pivot_table`, `crosstab`), часова динаміка та кореляції

In [7]:
global_q = (
    df.groupby(["period_date", "period_label", "year", "quarter", "language", "language_type", "is_general_purpose"], as_index=False)
    .agg(total_pushers=("num_pushers", "sum"), countries_count=("iso2_code", "nunique"))
    .sort_values(["language", "period_date"])
)
global_q["rolling_4q_pushers"] = global_q.groupby("language")["total_pushers"].transform(lambda s: s.rolling(4, min_periods=1).mean())

top12_gp = global_prog_2026[global_prog_2026["is_general_purpose"]].nlargest(12, "global_pushers")["language"].tolist()
pivot_years = global_q[(global_q["quarter"] == 2) & (global_q["language"].isin(top12_gp))].pivot_table(
    index="language", columns="year", values="total_pushers", aggfunc="sum"
).loc[top12_gp]
pivot_years["Зростання 2020→2026 (разів)"] = (pivot_years[2026] / pivot_years[2020]).round(2)
print("Зведена таблиця 1 — Динаміка Топ-12 загальнопризначених мов (Q2 кожного року):")
display(pivot_years)

pivot_era_tier = prog_latest.pivot_table(
    index="language_era", columns="dev_market_tier", values="pushers_per_1000_devs", aggfunc="mean", observed=True
).round(2)
print("\nЗведена таблиця 2 — Середня інтенсивність (pushers / 1000 devs) за поколінням мов і масштабом ринку:")
display(pivot_era_tier)

ct_type_era = pd.crosstab(df.drop_duplicates("language")["language_type"], df.drop_duplicates("language")["language_era"])
print("\nТаблиця спряженості (crosstab) — Кількість мов за типом і поколінням:")
display(ct_type_era)

# Кореляційний аналіз на рівні унікальних мов (2026 Q2 + PLDB)
lang_summary = pd.read_csv(PROCESSED_DIR / "languages_pldb_summary.csv")
corr_cols = [
    "global_pushers_2026q2", "countries_2026q2", "pldbScore",
    "numberOfUsersEstimate", "numberOfJobsEstimate",
    "wikipedia_dailyPageViews", "bookCount", "language_age_2026",
]
corr_spearman = lang_summary[corr_cols].dropna().corr(method="spearman").round(3)
print(f"\nКореляційна матриця Спірмена (n = {len(lang_summary[corr_cols].dropna())} мов):")
display(corr_spearman)

Зведена таблиця 1 — Динаміка Топ-12 загальнопризначених мов (Q2 кожного року):

year,2020,2021,2022,2023,2024,2025,2026,Зростання 2020→2026 (разів)
language,,,,,,,,
JavaScript,1621043,1862742,2208208,2564676,2911560,3749975,6162445,3.80
Python,951339,1069369,1226679,1507468,1800649,2572558,4509915,4.74
TypeScript,301871,398397,540404,753188,936169,1448980,3025534,10.02
Java,662528,663023,716059,788172,796768,882734,1062585,1.60
C++,361437,372609,448244,520188,586072,692511,930908,2.58
C,314603,340223,423398,514758,557483,666475,874642,2.78
PHP,326788,338980,359208,382650,418081,475571,547998,1.68
C#,257229,276893,316884,354597,383335,431260,525956,2.04
Kotlin,104230,139238,170505,208917,246283,299986,463932,4.45



Зведена таблиця 2 — Середня інтенсивність (pushers / 1000 devs) за поколінням мов і масштабом ринку:


dev_market_tier,Малі (<50 тис.),Середні (50–500 тис.),Великі (0.5–2 млн),Глобальні лідери (>2 млн)
language_era,,,,
До 1990 (класичні),6.36,3.64,1.97,1.11
1990–2004 (веб-епоха),17.71,7.54,3.68,1.77
2005–2014 (сучасні),9.41,4.02,2.06,1.03
2015+ (новітні),NaN,0.47,0.21,0.08



Таблиця спряженості (crosstab) — Кількість мов за типом і поколінням:


language_era,До 1990 (класичні),1990–2004 (веб-епоха),2005–2014 (сучасні),2015+ (новітні)
language_type,,,,
data,1,5,2,0
markup,3,4,25,3
programming,59,98,109,30
prose,0,2,0,0



Кореляційна матриця Спірмена (n = 202 мов):

,global_pushers_2026q2,countries_2026q2,pldbScore,numberOfUsersEstimate,numberOfJobsEstimate,wikipedia_dailyPageViews,bookCount,language_age_2026
global_pushers_2026q2,1.000,0.993,0.520,0.489,0.467,0.558,0.410,0.062
countries_2026q2,0.993,1.000,0.526,0.494,0.477,0.564,0.419,0.060
pldbScore,0.520,0.526,1.000,0.926,0.739,0.817,0.670,0.104
numberOfUsersEstimate,0.489,0.494,0.926,1.000,0.717,0.861,0.613,0.069
numberOfJobsEstimate,0.467,0.477,0.739,0.717,1.000,0.686,0.670,0.164
wikipedia_dailyPageViews,0.558,0.564,0.817,0.861,0.686,1.000,0.641,0.225
bookCount,0.410,0.419,0.670,0.613,0.670,0.641,1.000,0.275
language_age_2026,0.062,0.060,0.104,0.069,0.164,0.225,0.275,1.000


## 6. Візуалізація результатів засобами Matplotlib і Seaborn
Нижче наведено вісім аналітичних графіків, які послідовно відповідають на поставлені дослідницькі питання (від рейтингів і динаміки до географічних профілів та кореляційних зв’язків).

In [8]:
# Графік 1: Стовпчикова діаграма (Топ-12 світ vs Україна, 2026 Q2)
fig, axes = plt.subplots(1, 2, figsize=(14.5, 6.2))
g12 = global_prog_2026.nlargest(12, "global_pushers").iloc[::-1]
axes[0].barh(g12["language"], g12["global_pushers"] / 1e6, color=["#1f77b4" if gp else "#7f8c8d" for gp in g12["is_general_purpose"]])
axes[0].set_title("а) Світ: Топ-12 мов програмування (2026 Q2)")
axes[0].set_xlabel("Активні розробники (pushers), млн")
axes[0].set_xlim(0, g12["global_pushers"].max() / 1e6 * 1.18)

u12 = ua_comp.nlargest(12, "num_pushers").iloc[::-1]
axes[1].barh(u12["language"], u12["num_pushers"] / 1e3, color=["#2ca02c" if d >= 3 else "#1f77b4" for d in u12["rank_diff"]])
axes[1].set_title("б) Україна (UA): Топ-12 мов програмування (2026 Q2)")
axes[1].set_xlabel("Активні розробники (pushers), тис.")
axes[1].set_xlim(0, u12["num_pushers"].max() / 1e3 * 1.24)
plt.tight_layout()
plt.show()

# Графік 2: Лінійний графік квартальної динаміки Топ-8 загальнопризначених мов
top8_gp = ["JavaScript", "Python", "TypeScript", "Java", "C++", "C", "PHP", "C#"]
fig, ax = plt.subplots(figsize=(12.5, 5.8))
pal = sns.color_palette("tab10", n_colors=len(top8_gp))
for idx, lang in enumerate(top8_gp):
    sub = global_q[global_q["language"] == lang].sort_values("period_date")
    ax.plot(sub["period_date"], sub["total_pushers"] / 1e6, color=pal[idx], alpha=0.32, linewidth=1.2)
    ax.plot(sub["period_date"], sub["rolling_4q_pushers"] / 1e6, color=pal[idx], linewidth=2.4, label=lang)
ax.set_title("Квартальна динаміка Топ-8 загальнопризначених мов (2020–2026 рр.) із 4-кв. ковзним середнім")
ax.set_xlabel("Період")
ax.set_ylabel("Активні розробники у світі, млн")
ax.legend(ncol=4, frameon=True)
plt.tight_layout()
plt.show()

C:\Users\nazar\AppData\Local\Temp\ipykernel_8996\1521704089.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\nazar\AppData\Local\Temp\ipykernel_8996\1521704089.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# Графік 3: Темпи зростання (2020 Q1 -> 2026 Q2)
growth_df = pd.read_csv(PROJECT_ROOT / "results" / "exports" / "growth_2020_2026.csv")
focus_14 = ["TypeScript", "Rust", "Dart", "Python", "Kotlin", "Swift", "JavaScript", "Go", "C++", "C#", "C", "PHP", "Java", "Ruby"]
g14 = growth_df[growth_df["language"].isin(focus_14)].sort_values("growth_multiple")
fig, ax = plt.subplots(figsize=(11.2, 5.8))
cols_g = ["#2ca02c" if m >= 5.0 else ("#1f77b4" if m >= 2.5 else "#e67e22") for m in g14["growth_multiple"]]
bars = ax.barh(g14["language"], g14["growth_multiple"], color=cols_g)
ax.axvline(1.0, color="#555", linestyle="--", linewidth=1)
ax.set_title("Кратність зростання активних розробників за період 2020 Q1 – 2026 Q2")
ax.set_xlabel("Кратність зростання (2026 Q2 / 2020 Q1), разів")
for bar, val in zip(bars, g14["growth_multiple"]):
    ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2, f"×{val:.2f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

# Графік 4 і 5: Гістограми розподілу та діаграми розмаху (boxplot)
fig, axes = plt.subplots(1, 2, figsize=(13.6, 5.0))
sns.histplot(np.log10(df["num_pushers"]), bins=35, kde=True, color="#1f77b4", ax=axes[0])
axes[0].set_title("а) Логарифмічний розподіл log10(num_pushers)")
sns.boxplot(data=df.assign(log10_pushers=np.log10(df["num_pushers"])), x="language_type", y="log10_pushers", order=["programming", "markup", "data", "prose"], ax=axes[1], fliersize=2)
axes[1].set_title("б) Діаграма розмаху за типом мови (log10)")
plt.tight_layout()
plt.show()

C:\Users\nazar\AppData\Local\Temp\ipykernel_8996\1214158183.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\nazar\AppData\Local\Temp\ipykernel_8996\1214158183.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# Графік 6 і 7: Теплові карти кореляції Спірмена та географічного профілю (10 країн)
fig, ax = plt.subplots(figsize=(9.2, 7.2))
sns.heatmap(corr_spearman, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Кореляційна матриця Спірмена (GitHub 2026 Q2 + PLDB)", pad=14)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

selected_isos = ["US", "IN", "CN", "DE", "GB", "BR", "JP", "PL", "CA", "UA"]
top10_gp = ["JavaScript", "Python", "TypeScript", "Java", "C++", "C", "PHP", "C#", "Kotlin", "Go"]
pv_country = prog_latest[prog_latest["iso2_code"].isin(selected_isos) & prog_latest["language"].isin(top10_gp)].pivot_table(
    index="language", columns="iso2_code", values="language_share_pct", aggfunc="sum"
).reindex(index=top10_gp, columns=selected_isos)
fig, ax = plt.subplots(figsize=(11.2, 5.8))
sns.heatmap(pv_country, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax)
ax.set_title("Частка Топ-10 мов (%) у провідних економіках та Україні (2026 Q2)")
plt.tight_layout()
plt.show()

# Графік 8: Діаграма розсіювання GitHub pushers vs PLDB Jobs Estimate
scatter_df = lang_summary[(lang_summary["language_type"] == "programming") & (lang_summary["numberOfJobsEstimate"] > 0)].dropna(subset=["language_era"])
fig, ax = plt.subplots(figsize=(10.8, 5.8))
sns.scatterplot(data=scatter_df, x="global_pushers_2026q2", y="numberOfJobsEstimate", hue="language_era", size="countries_2026q2", sizes=(40, 320), alpha=0.82, ax=ax)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title(f"Зв'язок активності на GitHub (2026 Q2) та оцінки вакансій у PLDB (n = {len(scatter_df)} мов)")
ax.set_xlabel("Активні розробники на GitHub (лог-шкала)")
ax.set_ylabel("Оцінка вакансій у PLDB (лог-шкала)")
plt.tight_layout()
plt.show()

C:\Users\nazar\AppData\Local\Temp\ipykernel_8996\3357354505.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\nazar\AppData\Local\Temp\ipykernel_8996\3357354505.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\nazar\AppData\Local\Temp\ipykernel_8996\3357354505.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Підсумки аналізу в Jupyter Notebook
1. **Домінуюче ядро та інструменти збірки:** `JavaScript` (6.16 млн), `Python` (4.51 млн) і `TypeScript` (3.03 млн) забезпечують 44.03% усіх push-подій у категорії `programming` у 2026 Q2, водночас значну частку верхніх позицій займають інфраструктурні технології (`Shell`, `Dockerfile`, `Makefile`).
2. **Лідери зростання 2020–2026 рр.:** серед загальнопризначених мов із базою ≥10 тис. розробників у 2020 Q1 найшвидше зростали `PLpgSQL` (×17.00), `TypeScript` (×12.23), `Rust` (×11.08), `Dart` (×6.41) та `Python` (×5.55), тоді як зрілі екосистеми (`Ruby` ×1.37, `Java` ×1.76, `PHP` ×1.89) зростали помірніше.
3. **Специфіка України (`UA`):** загальна кількість облікових записів розробників зросла в **3.21 раза** (з 568.6 тис. до 1.83 млн). Мова **C#** посідає **6-ту позицію** в Україні (8 040 розробників, 3.90%) проти 13-ї у світі (+7 позицій), а **Dockerfile** піднімається на **4-ту позицію** (7.01%).
4. **Експорт для Tableau Public:** усі збагачені дані збережено до `tableau/tableau_coursework_data.xlsx`, а всі 8 графіків високої роздільності — до `results/figures/`.